In [1]:
!nvidia-smi

Sat Sep 26 13:32:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q unsloth
!pip install -q trl datasets transformers accelerate peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199

In [3]:
import torch
import json
import os

from datasets import Dataset, load_dataset
from unsloth import FastLanguageModel

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,

    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

Unsloth 2026.9.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": "How can I cancel my order?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 150,
    temperature = 0.7,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
How can I cancel my order?
assistant
I'm sorry to hear that you're experiencing issues with your order. Please follow the steps below to cancel it:

1. Log in to your account on the website or app where you made the purchase.
2. Go to the "My Orders" section and find the order you want to cancel.
3. Click on the "Cancel Order" button or option provided for cancellation.

If you're unable to locate these options, please try clearing your browser cache or restarting your device. If you continue to have trouble, consider reaching out to customer support through the appropriate channels for assistance. They should be able to guide you through the process of canceling your order.


In [7]:
raw_paragraphs = [
"""
NovaCart allows customers to cancel orders before they have been shipped.
Customers can open the Orders section of their account, select the relevant
order, and choose the Cancel Order option. Once an order has shipped,
cancellation is no longer available and the customer may instead request
a return after delivery.
""",

"""
NovaCart customers may return eligible products within 30 days of delivery.
The product should be returned in its original condition whenever possible.
Customers can initiate a return through the Orders section by selecting
the product and choosing the Return option.
""",

"""
Refunds for approved returns are generally processed within 5 to 7 business
days after the returned product has been inspected. The refund is normally
sent to the original payment method used for the purchase.
""",

"""
Customers who receive a damaged product should report the issue within
48 hours of delivery. They should provide the order number and photographs
showing the damage. NovaCart may provide a replacement or refund after
reviewing the request.
""",

"""
If a customer receives the wrong product, the issue should be reported
through the Help Center. Customers should provide the order number and
details of the incorrect item. NovaCart can arrange a replacement or
return for eligible cases.
""",

"""
Orders placed through NovaCart can normally be tracked through the Orders
section of the customer's account. Once the package has shipped, tracking
information becomes available under the Track Order option.
""",

"""
Standard NovaCart deliveries typically take between 3 and 7 business days.
Actual delivery time may vary depending on customer location, product
availability, courier capacity, and local conditions.
""",

"""
If a payment fails during checkout, customers should verify their payment
details and available balance before attempting the transaction again.
They may also try another supported payment method.
""",

"""
If a customer sees a duplicate payment charge, the customer should first
check whether both transactions have been completed. The customer may
contact NovaCart support and provide the transaction ID and order number
for investigation.
""",

"""
Customers who receive a package with missing items should report the issue
within 48 hours of delivery. The order number, product details, and package
information should be provided to customer support.
""",

"""
Some products sold through NovaCart include a manufacturer warranty.
Warranty duration and coverage depend on the individual product and brand.
Customers should check the warranty information listed on the product page.
""",

"""
A delivery address can be changed before an order has shipped. Customers
should update the address from their Orders section when the option is
available. Once shipping begins, the delivery address may no longer be changed.
""",

"""
NovaCart promotional coupons must normally be entered during checkout.
A coupon cannot usually be applied after an order has already been placed.
Coupon eligibility may depend on products, minimum purchase amount, and
promotion conditions.
""",

"""
Customers who forget their NovaCart password can select the Forgot Password
option on the sign-in page. A password reset link or verification process
will then be provided to the registered account.
""",

"""
NovaCart provides customer assistance through its Help Center. Customers
should include relevant information such as order number, transaction ID,
product details, and a description of the issue when submitting requests.
"""
]

# Expand the policies into additional wording/context
topics = raw_paragraphs.copy()

additional_contexts = [
    "Customers should always keep their order number available when contacting support.",
    "Providing complete information helps NovaCart resolve support requests faster.",
    "Customers should use their registered account when managing orders.",
]

while len(raw_paragraphs) < 60:
    base = topics[len(raw_paragraphs) % len(topics)]
    extra = additional_contexts[len(raw_paragraphs) % len(additional_contexts)]

    raw_paragraphs.append(
        base.strip() + "\n\n" + extra
    )

print("Number of paragraphs:", len(raw_paragraphs))

os.makedirs("data", exist_ok=True)

with open("data/non_instruction_data.txt", "w", encoding="utf-8") as f:
    for paragraph in raw_paragraphs:
        f.write(paragraph.strip())
        f.write("\n\n")

print("Saved!")

Number of paragraphs: 60
Saved!


In [8]:
dataset = Dataset.from_dict({
    "text": [p.strip() for p in raw_paragraphs]
})

dataset

Dataset({
    features: ['text'],
    num_rows: 60
})

In [9]:
print(dataset[0]["text"])

NovaCart allows customers to cancel orders before they have been shipped.
Customers can open the Orders section of their account, select the relevant
order, and choose the Cancel Order option. Once an order has shipped,
cancellation is no longer available and the customer may instead request
a return after delivery.


In [10]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir = "outputs_non_instruction",

    # Dataset
    dataset_text_field = "text",
    max_length = 1024,

    # Training
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    num_train_epochs = 2,

    # Optimizer
    learning_rate = 2e-4,
    warmup_ratio = 0.1,
    weight_decay = 0.01,

    # Precision
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),

    # Logging
    logging_steps = 1,

    # Optimizer suitable for quantized training
    optim = "adamw_8bit",

    seed = 42,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/60 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [11]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_availa

Step,Training Loss
1,3.023914
2,3.229675
3,3.240505
4,3.154497
5,2.916012
6,2.661540
7,2.744774
8,2.453291
9,2.530126
10,2.506665


Unsloth: Restored added_tokens_decoder metadata in outputs_non_instruction/checkpoint-30/tokenizer_config.json.


TrainOutput(global_step=30, training_loss=1.8730458637078604, metrics={'train_runtime': 37.5896, 'train_samples_per_second': 3.192, 'train_steps_per_second': 0.798, 'total_flos': 47677536890880.0, 'train_loss': 1.8730458637078604, 'epoch': 2.0})


In [12]:
trainer.state.log_history

[{'loss': 3.023913860321045,
  'grad_norm': 1.3581583499908447,
  'learning_rate': 0.0,
  'epoch': 0.06666666666666667,
  'step': 1},
 {'loss': 3.22967529296875,
  'grad_norm': 1.265798568725586,
  'learning_rate': 6.666666666666667e-05,
  'epoch': 0.13333333333333333,
  'step': 2},
 {'loss': 3.2405052185058594,
  'grad_norm': 1.4207484722137451,
  'learning_rate': 0.00013333333333333334,
  'epoch': 0.2,
  'step': 3},
 {'loss': 3.154496669769287,
  'grad_norm': 1.1234383583068848,
  'learning_rate': 0.0002,
  'epoch': 0.26666666666666666,
  'step': 4},
 {'loss': 2.9160118103027344,
  'grad_norm': 2.172785520553589,
  'learning_rate': 0.0001925925925925926,
  'epoch': 0.3333333333333333,
  'step': 5},
 {'loss': 2.6615397930145264,
  'grad_norm': 1.2218841314315796,
  'learning_rate': 0.0001851851851851852,
  'epoch': 0.4,
  'step': 6},
 {'loss': 2.744774341583252,
  'grad_norm': 1.3868331909179688,
  'learning_rate': 0.00017777777777777779,
  'epoch': 0.4666666666666667,
  'step': 7},
 

In [13]:
stage1_path = "models/novacart_non_instruction"

os.makedirs("models", exist_ok=True)

model.save_pretrained(stage1_path)
tokenizer.save_pretrained(stage1_path)

print("Stage 1 model saved to:", stage1_path)

Unsloth: Restored added_tokens_decoder metadata in models/novacart_non_instruction/tokenizer_config.json.


Stage 1 model saved to: models/novacart_non_instruction


In [14]:
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": "How long does NovaCart take to process a refund?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 100,
    temperature = 0.7,
)

response = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens = True
)

print(response)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The processing time for refunds can vary depending on the specific circumstances and the current demand at the store. Generally speaking, customers should expect their refund to be processed within 5-7 business days after submitting the request. However, it's always best to check the customer support page of the relevant store or contact them directly if there is no update after this period. They will provide an estimated timeframe based on the situation.


In [15]:
stage1_config = {
    "model": "Qwen2.5-1.5B-Instruct",
    "training_type": "Non-instruction QLoRA",
    "dataset_size": len(dataset),
    "epochs": 2,
    "batch_size": 2,
    "gradient_accumulation_steps": 4,
    "effective_batch_size": 8,
    "learning_rate": 2e-4,
    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,
    "max_seq_length": 1024,
    "load_in_4bit": True,
}

stage1_config

{'model': 'Qwen2.5-1.5B-Instruct',
 'training_type': 'Non-instruction QLoRA',
 'dataset_size': 60,
 'epochs': 2,
 'batch_size': 2,
 'gradient_accumulation_steps': 4,
 'effective_batch_size': 8,
 'learning_rate': 0.0002,
 'lora_rank': 16,
 'lora_alpha': 16,
 'lora_dropout': 0,
 'max_seq_length': 1024,
 'load_in_4bit': True}

In [16]:
import json

os.makedirs("reports", exist_ok=True)

with open("reports/stage1_training_config.json", "w") as f:
    json.dump(stage1_config, f, indent=4)

print("Configuration saved.")

Configuration saved.


In [17]:
training_logs = trainer.state.log_history

with open("reports/stage1_training_logs.json", "w") as f:
    json.dump(training_logs, f, indent=4)

print("Training logs saved.")

Training logs saved.


In [18]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [19]:
import os

PROJECT_DIR = "/content/drive/MyDrive/novacart-domain-ai-assistant"

DATA_DIR = f"{PROJECT_DIR}/data"
MODEL_DIR = f"{PROJECT_DIR}/models"
REPORT_DIR = f"{PROJECT_DIR}/reports"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)

Project directory: /content/drive/MyDrive/novacart-domain-ai-assistant


In [20]:
stage1_drive_path = f"{MODEL_DIR}/novacart-qwen-stage1-domain"

model.save_pretrained(stage1_drive_path)
tokenizer.save_pretrained(stage1_drive_path)

print("Stage 1 adapter saved to:")
print(stage1_drive_path)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/novacart-domain-ai-assistant/models/novacart-qwen-stage1-domain/tokenizer_config.json.


Stage 1 adapter saved to:
/content/drive/MyDrive/novacart-domain-ai-assistant/models/novacart-qwen-stage1-domain


In [21]:
non_instruction_file = f"{DATA_DIR}/non_instruction_data.txt"

with open(non_instruction_file, "w", encoding="utf-8") as f:
    for paragraph in raw_paragraphs:
        f.write(paragraph.strip())
        f.write("\n\n")

print("Dataset saved to:")
print(non_instruction_file)

Dataset saved to:
/content/drive/MyDrive/novacart-domain-ai-assistant/data/non_instruction_data.txt


In [22]:
import json

stage1_config = {
    "base_model": "unsloth/Qwen2.5-1.5B-Instruct",
    "training_stage": "Non-Instruction Fine-Tuning",
    "training_method": "QLoRA",

    "dataset_size": len(dataset),

    "epochs": 2,

    "per_device_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "effective_batch_size": 8,

    "learning_rate": 2e-4,

    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,

    "max_seq_length": 1024,

    "quantization": "4-bit",
}

config_path = f"{REPORT_DIR}/stage1_training_config.json"

with open(config_path, "w") as f:
    json.dump(stage1_config, f, indent=4)

print("Training config saved:", config_path)

Training config saved: /content/drive/MyDrive/novacart-domain-ai-assistant/reports/stage1_training_config.json


In [23]:
training_logs = trainer.state.log_history

logs_path = f"{REPORT_DIR}/stage1_training_logs.json"

with open(logs_path, "w") as f:
    json.dump(training_logs, f, indent=4)

print("Training logs saved:", logs_path)

Training logs saved: /content/drive/MyDrive/novacart-domain-ai-assistant/reports/stage1_training_logs.json


In [24]:
!pip install -q huggingface_hub

In [38]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face token: ")

Enter your Hugging Face token: ··········


In [42]:
HF_USERNAME = "Devu758"

HF_REPO_STAGE1 = f"{HF_USERNAME}/novacart-qwen-stage1-domain"

print(HF_REPO_STAGE1)

Devu758/novacart-qwen-stage1-domain


In [43]:
model.push_to_hub(
    HF_REPO_STAGE1,
    private=False,
)

tokenizer.push_to_hub(
    HF_REPO_STAGE1,
)

print("Stage 1 pushed to Hugging Face:")
print(f"https://huggingface.co/{HF_REPO_STAGE1}")

README.md:   0%|          | 0.00/590 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Saved model to https://huggingface.co/Devu758/novacart-qwen-stage1-domain


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpl6ryxfcu/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpl6ryxfcu/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Stage 1 pushed to Hugging Face:
https://huggingface.co/Devu758/novacart-qwen-stage1-domain
